In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

base = "/Volumes/data-bricks-fundamentals/default/average-price-price"   # change catalog name if yours differs

def statcan_schema(product_col):
    return StructType([
        StructField("ref_date", StringType()),
        StructField("geo", StringType()),
        StructField("dguid", StringType()),
        StructField(product_col, StringType()),
        StructField("uom", StringType()),
        StructField("uom_id", IntegerType()),
        StructField("scalar_factor", StringType()),
        StructField("scalar_id", IntegerType()),
        StructField("vector", StringType()),
        StructField("coordinate", StringType()),
        StructField("value", DoubleType()),
        StructField("status", StringType()),
        StructField("symbol", StringType()),
        StructField("terminated", StringType()),
        StructField("decimals", IntegerType()),
    ])

avg_df = (spark.read
    .option("header", "true")
    .schema(statcan_schema("products"))
    .csv(f"{base}/avg/18100245.csv"))

cpi_df = (spark.read
    .option("header", "true")
    .schema(statcan_schema("products_and_product_groups"))
    .csv(f"{base}/provience-wise/18100004.csv"))

# Checks
print("avg rows:", avg_df.count())   # expect 141,432
print("cpi rows:", cpi_df.count())    # expect 1,154,918

display(avg_df.limit(10))
display(cpi_df.limit(10))
display(avg_df.groupBy("geo").count().orderBy("geo"))
display(cpi_df.groupBy("geo").count().orderBy("geo"))
display(avg_df.agg(F.min("ref_date"), F.max("ref_date")))

In [0]:
def to_bronze(df, table_name):
    (df
     .withColumn("_ingested_at", F.current_timestamp())
     .withColumn("_source_file", F.col("_metadata.file_name"))
     .write
     .mode("overwrite")
     .saveAsTable(f"`data-bricks-fundamentals`.default.{table_name}"))

to_bronze(avg_df, "bronze_avg_prices")
to_bronze(cpi_df, "bronze_cpi")

In [0]:
%sql
SELECT 'avg_prices' AS tbl, COUNT(*) AS rows FROM `data-bricks-fundamentals`.default.bronze_avg_prices
UNION ALL
SELECT 'cpi', COUNT(*) FROM `data-bricks-fundamentals`.default.bronze_cpi;